In [ ]:
# Run this command if pyspark is not installed or getting this error: ImportError: No module named pyspark
!pip install pyspark


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=8a096899eccd9f93fb9942bec14e736d1fb3256bd6d04eec0ae8e1f9cea1503c
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [ ]:
# importing all the python packages
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import countDistinct
from pyspark.sql.functions import col, max


# creating spark session to read csv files
spark = SparkSession.builder.appName("Read CSV").getOrCreate()
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")


In [ ]:
# creating respective spark dataframe and loading data from respective csv files
carsFile = spark.read.csv("CARS.csv", header=True, inferSchema=True)

customersFile = spark.read.csv("CUSTOMERS.csv", header=True, inferSchema=True)

householdsFile = spark.read.csv("HOUSEHOLDS.csv", header=True, inferSchema=True)

In [ ]:
# checking schema for carsFile DF
carsFile.printSchema()

root
 |-- Car ID: integer (nullable = true)
 |-- Status: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Make: string (nullable = true)
 |-- Body Style: string (nullable = true)
 |-- Vehicle Value: double (nullable = true)
 |-- Annual Miles Driven: integer (nullable = true)
 |-- Business Use: integer (nullable = true)
 |-- Antique Vehicle: integer (nullable = true)
 |-- Lien: integer (nullable = true)
 |-- Lease: integer (nullable = true)
 |-- Driver Safety Discount: integer (nullable = true)
 |-- Vehicle Safety Discount: integer (nullable = true)
 |-- Claim Payout: integer (nullable = true)
 |-- 6 Month Premium Amount: double (nullable = true)



In [ ]:
carsFile.select("CAR_CAR_ID","Status","CAR_State").orderBy("CAR_CAR_ID").show()

+----------+--------------------+---------+
|CAR_CAR_ID|              Status|CAR_State|
+----------+--------------------+---------+
|       386|            In Force|       CA|
|       402|Customer Cancella...|       NC|
|       555|            In Force|       IN|
|       789|            In Force|       MO|
|       897|            In Force|       TN|
|      1000|            In Force|       IL|
|      1001|            In Force|       IN|
|      1001|            In Force|       IL|
|      1001|            In Force|       MT|
|      1002|            In Force|       AZ|
|      1002|            In Force|       ME|
|      1003|            In Force|       ND|
|      1003|            In Force|       AK|
|      1004|Customer Cancella...|       MD|
|      1004|            In Force|       MO|
|      1004|            In Force|       AK|
|      1004|Customer Cancella...|       IA|
|      1004|            In Force|       VA|
|      1005|            In Force|       MO|
|      1005|            In Force

In [ ]:
# checking schema for customersFile DF
customersFile.printSchema()

root
 |-- CUST_ID: integer (nullable = true)
 |-- Date of Birth: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marital Status: string (nullable = true)
 |-- Employment Type: string (nullable = true)
 |-- Income: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)



In [ ]:
# checking schema for householdsFile DF
householdsFile.printSchema()

root
 |-- HH_ID: integer (nullable = true)
 |-- CUST_ID: integer (nullable = true)
 |-- CAR_ID: integer (nullable = true)
 |-- Active HH: integer (nullable = true)
 |-- HH Start Date: string (nullable = true)
 |-- Phone Number: string (nullable = true)
 |-- ZIP : integer (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Referral Source: string (nullable = true)



In [ ]:
# Checking in CustomersFile dataframe if there are any columns that have all the null values in it
print("Counts of rows in Customers File : "+str(customersFile.count()))
print("*********************************")
null_CF_counts = customersFile.select([F.count(F.when(F.col(c).isNull(), c)).alias(
    c) for c in customersFile.columns]).collect()[0].asDict()

print("List of count of null values for each columns in Customers File:")
print(null_CF_counts)
print("*********************************")
customersFile_size = customersFile.count()

CFcolumns_to_drop = [k for k, v in null_CF_counts.items() if v == customersFile_size]

print("List of columns in Customers File that have all values as null:")
print(CFcolumns_to_drop)
print("*********************************")
customersFile = customersFile.drop(*CFcolumns_to_drop)

print("Counts of rows in Customers File after removing the columns: "+str(customersFile.count()))

Counts of rows in Customers File : 499999
*********************************
List of count of null values for each columns in Customers File:
{'CUST_ID': 0, 'Date of Birth': 0, 'Gender': 0, 'Marital Status': 0, 'Employment Type': 0, 'Income': 0, '_c6': 499999, '_c7': 499999, '_c8': 499999, '_c9': 499999, '_c10': 499999, '_c11': 499999, '_c12': 499999, '_c13': 499999, '_c14': 499999, '_c15': 499999}
*********************************
List of columns in Customers File that have all values as null:
['_c6', '_c7', '_c8', '_c9', '_c10', '_c11', '_c12', '_c13', '_c14', '_c15']
*********************************
Counts of rows in Customers File after removing the columns: 499999


In [ ]:
# Checking in CarsFile if there are any columns that have all the null values in it
print("Counts of rows in Cars File : "+str(carsFile.count()))
print("*********************************")
null_cars_counts = carsFile.select([F.count(F.when(F.col(c).isNull(), c)).alias(
    c) for c in carsFile.columns]).collect()[0].asDict()

print("List of count of null values for each columns in Cars File")
print(null_cars_counts)
print("*********************************")

carsFile_size = carsFile.count()

CarsColumns_to_drop = [k for k, v in null_cars_counts.items() if v == carsFile_size]

print("List of columns in Cars File that have all values as null")
print(CarsColumns_to_drop)
print("*********************************")

carsFile = carsFile.drop(*CarsColumns_to_drop)

print("Counts of rows in Cars File after removing the columns")
carsFile.count()

Counts of rows in Cars File : 500000
*********************************
List of count of null values for each columns in Cars File
{'Car ID': 0, 'Status': 0, 'State': 0, 'Year': 0, 'Make': 0, 'Body Style': 0, 'Vehicle Value': 0, 'Annual Miles Driven': 0, 'Business Use': 0, 'Antique Vehicle': 0, 'Lien': 0, 'Lease': 0, 'Driver Safety Discount': 0, 'Vehicle Safety Discount': 0, 'Claim Payout': 0, '6 Month Premium Amount': 0}
*********************************
List of columns in Cars File that have all values as null
[]
*********************************
Counts of rows in Cars File after removing the columns


500000

In [ ]:
# Checking in HouseHoldsFile dataframe if there are any columns that have all the null values in it
print("Counts of rows in HouseHolds File : "+str(householdsFile.count()))
print("*********************************")
null_HH_counts = householdsFile.select([F.count(F.when(F.col(c).isNull(), c)).alias(
    c) for c in householdsFile.columns]).collect()[0].asDict()

print("List of count of null values for each columns in HouseHolds File")
print(null_HH_counts)
print("*********************************")

HHFile_size = householdsFile.count()

HHColumns_to_drop = [k for k, v in null_HH_counts.items() if v == HHFile_size]

print("List of columns in HouseHolds File that have all values as null")
print(HHColumns_to_drop)
print("*********************************")

householdsFile = householdsFile.drop(*HHColumns_to_drop)

print("Counts of rows in HouseHolds File after removing the columns")
householdsFile.count()


Counts of rows in HouseHolds File : 500000
*********************************
List of count of null values for each columns in HouseHolds File
{'HH_ID': 0, 'CUST_ID': 0, 'CAR_ID': 0, 'Active HH': 0, 'HH Start Date': 0, 'Phone Number': 0, 'ZIP ': 0, 'State': 0, 'Country': 0, 'Referral Source': 0}
*********************************
List of columns in HouseHolds File that have all values as null
[]
*********************************
Counts of rows in HouseHolds File after removing the columns


500000

In [ ]:
print("The count of rows in CARS.csv file: "+str(carsFile.count()))

print("The count of rows in CUSTOMERS.csv file: "+str(customersFile.count()))

print("The count of rows in HOUSEHOLDS.csv file: "+str(householdsFile.count()))

The count of rows in CARS.csv file: 500000
The count of rows in CUSTOMERS.csv file: 499999
The count of rows in HOUSEHOLDS.csv file: 500000


In [ ]:
#Renaming the Car Id to CAR_CAR_ID and State to CAR_STATE in CarsFile
carsFile = carsFile.withColumnRenamed("Car Id", "CAR_CAR_ID") \
                   .withColumnRenamed("State", "CAR_State")

#Renaming the CUST_ID to CUST_CUST_ID and State to CAR_STATE in CarsFile
customersFile = customersFile.withColumnRenamed("CUST_Id", "CUST_CUST_ID")

In [ ]:
# creating a dataframe to get the distinct count of CAR_ID in carsFile
distinctCarId = carsFile.select(countDistinct("CAR_CAR_ID"))
print("The distinct count of cars in CARS.csv file")
distinctCarId.show()

# creating a dataframe to get the distinct count of CUST_ID in customersFile
distinctCustomerId = customersFile.select(countDistinct("CUST_CUST_ID"))
print("The distinct count of customer in CUSTOMERS.csv file")
distinctCustomerId.show()

The distinct count of cars in CARS.csv file
+--------------------------+
|count(DISTINCT CAR_CAR_ID)|
+--------------------------+
|                    372536|
+--------------------------+

The distinct count of customer in CUSTOMERS.csv file
+----------------------------+
|count(DISTINCT CUST_CUST_ID)|
+----------------------------+
|                      499850|
+----------------------------+



In [ ]:
# creating a dataframe to get the distinct count of CUST_ID in customersFile
distinctCustomerId = householdsFile.select(countDistinct("CUST_ID"))
print("The distinct count of customer in HOUSEHOLDS.csv file")
distinctCustomerId.show()

# creating a dataframe to get the distinct count of CAR_ID in carsFile
distinctCarId = householdsFile.select(countDistinct("CAR_ID"))
print("The distinct count of car in HOUSEHOLDS.csv file")
distinctCarId.show()

The distinct count of customer in HOUSEHOLDS.csv file
+-----------------------+
|count(DISTINCT CUST_ID)|
+-----------------------+
|                 499851|
+-----------------------+

The distinct count of car in HOUSEHOLDS.csv file
+----------------------+
|count(DISTINCT CAR_ID)|
+----------------------+
|                372536|
+----------------------+



In [ ]:
# In households File there is one extra CUST_ID compared to CustomersFile. Below code is to find that CUST_ID
distHHCustId = householdsFile.orderBy("CUST_ID", ascending=False).select("CUST_ID")
#print(distHHCustId.count())


distCustId = customersFile.orderBy("CUST_CUST_ID", ascending=False).select("CUST_CUST_ID")
#print(distCustId.count())

diff_df = distHHCustId.subtract(distCustId)
print("The CUST_ID that is in householdsFile but not in CustomersFile")
diff_df.show()

The CUST_ID that is in householdsFile but not in CustomersFile
+---------+
|  CUST_ID|
+---------+
|774461007|
+---------+



In [ ]:
#Checking to make sure the CUST_ID found really exist in householdsFile but not in CustomersFile
householdsFile.filter("CUST_ID == '774461007'").show()

customersFile.filter("CUST_CUST_ID == '774461007'").show()

+---------+---------+------+---------+-------------+--------------+-----+-----+-------+---------------+
|    HH_ID|  CUST_ID|CAR_ID|Active HH|HH Start Date|  Phone Number| ZIP |State|Country|Referral Source|
+---------+---------+------+---------+-------------+--------------+-----+-----+-------+---------------+
|739569792|774461007|640160|        0|       8/1/81|(678) 123-4567|54321|   GA|    USA|          Other|
+---------+---------+------+---------+-------------+--------------+-----+-----+-------+---------------+

+------------+-------------+------+--------------+---------------+------+
|CUST_CUST_ID|Date of Birth|Gender|Marital Status|Employment Type|Income|
+------------+-------------+------+--------------+---------------+------+
+------------+-------------+------+--------------+---------------+------+



In [ ]:
# creating a dataframe df that is joined on householdsFile and CarsFile over CAR_ID and State
join_cond = [householdsFile.CAR_ID == carsFile.CAR_CAR_ID, householdsFile.State == carsFile.CAR_State]

df = householdsFile.join(carsFile, on=join_cond)
print("Count of df: "+str(df.count()))
df.printSchema()

Count of df: 500000
root
 |-- HH_ID: integer (nullable = true)
 |-- CUST_ID: integer (nullable = true)
 |-- CAR_ID: integer (nullable = true)
 |-- Active HH: integer (nullable = true)
 |-- HH Start Date: string (nullable = true)
 |-- Phone Number: string (nullable = true)
 |-- ZIP : integer (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Referral Source: string (nullable = true)
 |-- CAR_CAR_ID: integer (nullable = true)
 |-- Status: string (nullable = true)
 |-- CAR_State: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Make: string (nullable = true)
 |-- Body Style: string (nullable = true)
 |-- Vehicle Value: double (nullable = true)
 |-- Annual Miles Driven: integer (nullable = true)
 |-- Business Use: integer (nullable = true)
 |-- Antique Vehicle: integer (nullable = true)
 |-- Lien: integer (nullable = true)
 |-- Lease: integer (nullable = true)
 |-- Driver Safety Discount: integer (nullable = true)
 |-- Vehic

In [ ]:
# In CustomersFile converting 'Date of Birth' string type column to Date Type
customersFile = customersFile.withColumn('DOB',F.to_date(F.col('Date of Birth'), 'MM/dd/yyyy'))

#selecting the rows of cutomers with max DOB in the dataframe custFile
custFile = (customersFile.groupBy("CUST_CUST_ID").agg(max("DOB")))

#renaming columns in custFile
custFile = custFile.withColumnRenamed("max(DOB)", "max_DOB") \
                   .withColumnRenamed("CUST_CUST_ID", "UNI_CUST_ID")

print("The count of rows in custFile which is same as distinct count of customersId:"+str(custFile.count()))

join_cond = [custFile.UNI_CUST_ID == customersFile.CUST_CUST_ID, custFile.max_DOB == customersFile.DOB]

customersFile = customersFile.join(custFile, on=join_cond)
customersFile = customersFile.drop("UNI_CUST_ID","max_DOB")
print("Final customersFile after getting the customers Id with max DOB")
customersFile.printSchema()





The count of rows in custFile which is same as distinct count of customersId:499850
Final customersFile after getting the customers Id with max DOB
root
 |-- CUST_CUST_ID: integer (nullable = true)
 |-- Date of Birth: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marital Status: string (nullable = true)
 |-- Employment Type: string (nullable = true)
 |-- Income: string (nullable = true)
 |-- DOB: date (nullable = true)



In [ ]:
# now joining the customersFile with the df to combine all the files
join_cond = [df.CUST_ID == customersFile.CUST_CUST_ID]

df = df.join(customersFile, on=join_cond, how = "outer")


In [ ]:
# checking the columns of the dataframe
df.printSchema()

root
 |-- HH_ID: integer (nullable = true)
 |-- CUST_ID: integer (nullable = true)
 |-- CAR_ID: integer (nullable = true)
 |-- Active HH: integer (nullable = true)
 |-- HH Start Date: string (nullable = true)
 |-- Phone Number: string (nullable = true)
 |-- ZIP : integer (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Referral Source: string (nullable = true)
 |-- CAR_CAR_ID: integer (nullable = true)
 |-- Status: string (nullable = true)
 |-- CAR_State: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Make: string (nullable = true)
 |-- Body Style: string (nullable = true)
 |-- Vehicle Value: double (nullable = true)
 |-- Annual Miles Driven: integer (nullable = true)
 |-- Business Use: integer (nullable = true)
 |-- Antique Vehicle: integer (nullable = true)
 |-- Lien: integer (nullable = true)
 |-- Lease: integer (nullable = true)
 |-- Driver Safety Discount: integer (nullable = true)
 |-- Vehicle Safety Discount: 

In [ ]:
#Checking the distinct count of customer id in the combined dataframe which matches with the distinct count from customersFile
distinctCust = df.select(countDistinct("CUST_ID"))
distinctCust.show()


+-----------------------+
|count(DISTINCT CUST_ID)|
+-----------------------+
|                 499851|
+-----------------------+



In [ ]:
# checking for the customer id that was in householdsFile but nor in customersFile,
# all the columns from customerFile are Null
df.where(col("CUST_ID") == 774461007).show()

+---------+---------+------+---------+-------------+--------------+-----+-----+-------+---------------+----------+--------+---------+----+-------------+----------+-------------+-------------------+------------+---------------+----+-----+----------------------+-----------------------+------------+----------------------+------------+-------------+------+--------------+---------------+------+----+
|    HH_ID|  CUST_ID|CAR_ID|Active HH|HH Start Date|  Phone Number| ZIP |State|Country|Referral Source|CAR_CAR_ID|  Status|CAR_State|Year|         Make|Body Style|Vehicle Value|Annual Miles Driven|Business Use|Antique Vehicle|Lien|Lease|Driver Safety Discount|Vehicle Safety Discount|Claim Payout|6 Month Premium Amount|CUST_CUST_ID|Date of Birth|Gender|Marital Status|Employment Type|Income| DOB|
+---------+---------+------+---------+-------------+--------------+-----+-----+-------+---------------+----------+--------+---------+----+-------------+----------+-------------+-------------------+-------

In [ ]:
#Checking the distinct count of cars id in the combined dataframe which matches
#with the distinct count of cars id from carsFile
distinctCar = df.select(countDistinct("CAR_ID"))
distinctCar.show()

+----------------------+
|count(DISTINCT CAR_ID)|
+----------------------+
|                372536|
+----------------------+



In [ ]:
print("Total count of rows in the combined dataframe: "+str(df.count()))


Total count of rows in the combined dataframe: 500000


In [ ]:
print("The schema of the combined dataframe")
df.printSchema()

The schema of the combined dataframe
root
 |-- HH_ID: integer (nullable = true)
 |-- CUST_ID: integer (nullable = true)
 |-- CAR_ID: integer (nullable = true)
 |-- Active HH: integer (nullable = true)
 |-- HH Start Date: string (nullable = true)
 |-- Phone Number: string (nullable = true)
 |-- ZIP : integer (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Referral Source: string (nullable = true)
 |-- CAR_CAR_ID: integer (nullable = true)
 |-- Status: string (nullable = true)
 |-- CAR_State: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Make: string (nullable = true)
 |-- Body Style: string (nullable = true)
 |-- Vehicle Value: double (nullable = true)
 |-- Annual Miles Driven: integer (nullable = true)
 |-- Business Use: integer (nullable = true)
 |-- Antique Vehicle: integer (nullable = true)
 |-- Lien: integer (nullable = true)
 |-- Lease: integer (nullable = true)
 |-- Driver Safety Discount: integer (nullable =

In [ ]:
# Removing the duplicate columns of CAR_ID, State and Cust_Id from the final dataframe
finalResult = df.drop("CAR_CAR_ID", "CAR_State","CUST_CUST_ID" )

In [ ]:
# Printing Schema of the final dataframe after combining the data from all the files
finalResult.printSchema()


root
 |-- HH_ID: integer (nullable = true)
 |-- CUST_ID: integer (nullable = true)
 |-- CAR_ID: integer (nullable = true)
 |-- Active HH: integer (nullable = true)
 |-- HH Start Date: string (nullable = true)
 |-- Phone Number: string (nullable = true)
 |-- ZIP : integer (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Referral Source: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Make: string (nullable = true)
 |-- Body Style: string (nullable = true)
 |-- Vehicle Value: double (nullable = true)
 |-- Annual Miles Driven: integer (nullable = true)
 |-- Business Use: integer (nullable = true)
 |-- Antique Vehicle: integer (nullable = true)
 |-- Lien: integer (nullable = true)
 |-- Lease: integer (nullable = true)
 |-- Driver Safety Discount: integer (nullable = true)
 |-- Vehicle Safety Discount: integer (nullable = true)
 |-- Claim Payout: integer (nullable = true)
 |-- 6 Month 

In [ ]:
#Saving the final df in parquet file

finalResult.write.mode("overwrite").parquet("FINAL_RESULT")